In [ ]:
# In[1]:

import os
import pandas as pd
import numpy as np
import pytz

# Reusable variables (reuse if already present in kernel)
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat04','Tomcat03',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
tz_name = 'Asia/Shanghai'  # UTC+8 as required

# Helper to safely load CSV if exists
def load_csv_if_exists(fname):
    return pd.read_csv(fname) if os.path.exists(fname) else None

# Load files
df_metric = load_csv_if_exists('metric.csv')
df_trace = load_csv_if_exists('trace.csv')
df_log = load_csv_if_exists('log.csv')
df_error = load_csv_if_exists('error_logs.csv')

# Validate mandatory files presence
if df_metric is None:
    raise FileNotFoundError("metric.csv not found in working directory.")
if df_trace is None:
    raise FileNotFoundError("trace.csv not found in working directory.")
if df_log is None:
    raise FileNotFoundError("log.csv not found in working directory.")

# Ensure timestamp and value columns types for metric/trace/log
for df in (df_metric, df_trace, df_log):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Filter to allowed components
df_metric = df_metric[df_metric['cmdb_id'].isin(components)].copy()
df_trace = df_trace[df_trace['cmdb_id'].isin(components)].copy()
df_log = df_log[df_log['cmdb_id'].isin(components)].copy()

# Aggregation helper using robust groupby aggregation
def aggregate_timeseries(df, name_col):
    assert 'cmdb_id' in df.columns and 'timestamp' in df.columns and 'value' in df.columns and name_col in df.columns
    grp = df.groupby(['cmdb_id', name_col])
    agg = grp.agg(
        count = ('value', 'count'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max'),
        min_value = ('value', 'min'),
        max_value = ('value', 'max'),
        p50 = ('value', lambda x: x.quantile(0.5)),
        p95 = ('value', lambda x: x.quantile(0.95))
    ).reset_index()
    agg = agg.rename(columns={name_col: 'KPI_name'})
    # Convert timestamps to timezone-aware datetimes (UTC->Asia/Shanghai), handling NA safely
    agg['min_ts_local'] = pd.to_datetime(agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz_name)
    agg['max_ts_local'] = pd.to_datetime(agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz_name)
    # Round numeric columns for compactness
    for col in ['min_value','max_value','p50','p95']:
        if col in agg.columns:
            agg[col] = pd.to_numeric(agg[col], errors='coerce').round(6)
    return agg

# Aggregate each file and keep full tables in kernel variables
metric_agg = aggregate_timeseries(df_metric, 'kpi_name')
trace_agg = aggregate_timeseries(df_trace, 'trace_name')
log_agg = aggregate_timeseries(df_log, 'log_name')

# Compact summaries: top 50 by count desc
metric_summary = metric_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id','KPI_name','count','min_ts_local','max_ts_local','min_value','max_value','p50','p95']
]
trace_summary = trace_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id','KPI_name','count','min_ts_local','max_ts_local','min_value','max_value','p50','p95']
]
log_summary = log_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id','KPI_name','count','min_ts_local','max_ts_local','min_value','max_value','p50','p95']
]

# error_logs.csv existence and basic info (compact)
if df_error is None:
    error_log_info = pd.Series({'exists': False, 'row_count': 0, 'earliest_ts_local': pd.NaT, 'latest_ts_local': pd.NaT})
else:
    # Ensure numeric timestamps, handle missing timestamps safely
    ts_series = pd.to_numeric(df_error.get('timestamp', pd.Series(dtype='float')), errors='coerce').dropna()
    row_count = int(df_error.shape[0])
    if ts_series.empty:
        earliest = pd.NaT
        latest = pd.NaT
    else:
        min_ts = int(ts_series.min())
        max_ts = int(ts_series.max())
        earliest = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz_name)
        latest = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz_name)
    error_log_info = pd.Series({'exists': True, 'row_count': row_count,
                                'earliest_ts_local': earliest, 'latest_ts_local': latest})

# Final compact outputs (these variables will be displayed)
metric_summary, trace_summary, log_summary, error_log_info

```
Out[1]:
```
```python
# Summarize the prior aggregation results in plain English (compact)
# Reuse existing variables: metric_summary, trace_summary, log_summary, metric_agg, trace_agg, log_agg, error_log_info

# Safely extract top-row info if available
def top_info(df):
    if df is None or df.shape[0] == 0:
        return None
    r = df.iloc[0]
    return {
        'cmdb_id': str(r['cmdb_id']),
        'kpi': str(r['KPI_name']),
        'count': int(r['count']),
        'min_ts': str(r['min_ts_local']),
        'max_ts': str(r['max_ts_local']),
        'p95': str(r['p95'])
    }

m_top = top_info(metric_summary)
t_top = top_info(trace_summary)
l_top = top_info(log_summary)

metric_groups = metric_agg.shape[0] if 'metric_agg' in globals() else 0
trace_groups = trace_agg.shape[0] if 'trace_agg' in globals() else 0
log_groups = log_agg.shape[0] if 'log_agg' in globals() else 0

# Error logs info
err_exists = bool(error_log_info.get('exists', False)) if 'error_log_info' in globals() else False
err_count = int(error_log_info.get('row_count', 0)) if 'error_log_info' in globals() else 0
err_earliest = error_log_info.get('earliest_ts_local', pd.NaT) if 'error_log_info' in globals() else pd.NaT
err_latest = error_log_info.get('latest_ts_local', pd.NaT) if 'error_log_info' in globals() else pd.NaT

summary = (
    f"Loaded and aggregated telemetry for the allowed components.\n\n"
    f"Aggregated groups computed:\n"
    f"- metric groups: {metric_groups}\n"
    f"- trace groups:  {trace_groups}\n"
    f"- log groups:    {log_groups}\n\n"
    f"For each group we computed count, min/max timestamps (UTC+8), min/max values, median (P50), and global P95.\n\n"
    f"Top examples (highest-count group in each summary):\n"
    f"- Metric: {m_top['cmdb_id']} | {m_top['kpi']} | count={m_top['count']} | range={m_top['min_ts']} to {m_top['max_ts']} | P95={m_top['p95']}\n"
    f"- Trace:  {t_top['cmdb_id']} | {t_top['kpi']} | count={t_top['count']} | range={t_top['min_ts']} to {t_top['max_ts']} | P95={t_top['p95']}\n"
    f"- Log:    {l_top['cmdb_id']} | {l_top['kpi']} | count={l_top['count']} | range={l_top['min_ts']} to {l_top['max_ts']} | P95={l_top['p95']}\n\n"
    f"error_logs.csv: exists={err_exists}, row_count={err_count}"
    + (f", earliest={err_earliest}, latest={err_latest}" if err_count>0 else ", no timestamped rows found")
    + "\n\n"
    f"Full aggregated tables are stored in the kernel variables: metric_agg, trace_agg, log_agg."
)

# Display the compact summary string variable as required
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           KPI_name  count              min_ts_local              max_ts_local     min_value     max_value           p50           p95
1543  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  6.007100e+06  6.008906e+06  6.008004e+06  6.008816e+06
1537  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
1621  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  6.007096e+06  6.008902e+06  6.008000e+06  6.008812e+06
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
1536  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  4.000000e-01  5.000000e-01  4.000000e-01  5.000000e-01
...        ...                                                ...    ...                       ...                       ...           ...           ...           ...           ...
1348  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1349  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1350  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1351  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1352  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 9 columns],       cmdb_id                           KPI_name  count              min_ts_local              max_ts_local  min_value  max_value        p50        p95
0        IG01           trace.self.duration_mean     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.12150   1.585875   0.191905   1.157668
1        IG01            trace.self.duration_p95     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.27900   3.912250   0.570500   3.278387
2        IG01               trace.self.row_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    8.00000  80.000000  32.000000  73.100000
33       MG01  trace.from_Tomcat02.duration_mean     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.01000   0.406000   0.108766   0.337912
15       IG02           trace.self.duration_mean     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.06895   1.481444   0.185448   0.633244
..        ...                                ...    ...                       ...                       ...        ...        ...        ...        ...
121  Tomcat01         trace.to_MG01.duration_p95     29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.01500   0.539200   0.245800   0.510380
116  Tomcat01          trace.from_IG02.row_count     29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    1.00000  10.000000   5.000000   9.200000
114  Tomcat01      trace.from_IG02.duration_mean     29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.02700   0.521000   0.183000   0.482867
122  Tomcat01            trace.to_MG01.row_count     29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    1.00000  12.000000   4.000000  10.000000
143  Tomcat03          trace.from_IG01.row_count     29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    1.00000  13.000000   4.000000  10.000000

[50 rows x 9 columns],      cmdb_id          KPI_name  count              min_ts_local              max_ts_local  min_value  max_value     p50      p95
0   Tomcat01   log.error_count     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       0.00       0.00   0.000   0.0000
1   Tomcat01  log.gc_pause_max     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       1.41       5.15   5.130   5.1400
2   Tomcat01     log.row_count     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       4.00      93.00  79.000  89.5000
3   Tomcat02   log.error_count     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       0.00       0.00   0.000   0.0000
5   Tomcat02     log.row_count     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       1.00     100.00  75.000  91.0000
14  apache02   log.error_count     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       0.00       0.00   0.000   0.0000
15  apache02     log.row_count     31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00       2.00      39.00  15.000  36.0000
4   Tomcat02  log.gc_pause_max     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       5.03       5.13   5.110   5.1300
13  apache01     log.row_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       1.00      47.00  17.500  37.5000
6   Tomcat03   log.error_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       0.00       0.00   0.000   0.0000
8   Tomcat03     log.row_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       1.00      24.00   7.000  22.2000
9   Tomcat04   log.error_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       0.00       0.00   0.000   0.0000
12  apache01   log.error_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       0.00       0.00   0.000   0.0000
11  Tomcat04     log.row_count     30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00       1.00      23.00   7.000  16.0000
7   Tomcat03  log.gc_pause_max      2 2021-03-04 06:14:00+08:00 2021-03-04 06:29:00+08:00       0.07       0.09   0.080   0.0890
10  Tomcat04  log.gc_pause_max      2 2021-03-04 06:12:00+08:00 2021-03-04 06:26:00+08:00       0.02       0.07   0.045   0.0675, exists               True
row_count               0
earliest_ts_local     NaT
latest_ts_local       NaT
dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing variables from kernel: df_metric, df_trace, df_log, metric_agg, trace_agg, log_agg
tz = pytz.timezone('Asia/Shanghai')

# Define incident window in UTC epoch seconds (timestamps in files are epoch seconds UTC)
start_local = tz.localize(datetime(2021,3,4,6,0,0))
end_local = tz.localize(datetime(2021,3,4,6,30,0))
start_utc_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_utc_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter raw rows to the incident window (inclusive)
metric_win = df_metric[(df_metric['timestamp'] >= start_utc_epoch) & (df_metric['timestamp'] <= end_utc_epoch)].copy()
trace_win  = df_trace[(df_trace['timestamp']  >= start_utc_epoch) & (df_trace['timestamp']  <= end_utc_epoch)].copy()
log_win    = df_log[(df_log['timestamp']    >= start_utc_epoch) & (df_log['timestamp']    <= end_utc_epoch)].copy()

# Helper to compute global thresholds (p95 and p5) per group from full series (rule 9)
def compute_thresholds_full(df, name_col):
    grp = df.groupby(['cmdb_id', name_col])['value']
    thresh = grp.agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()
    thresh = thresh.rename(columns={name_col: 'KPI_name'})
    return thresh

metric_thresh = compute_thresholds_full(df_metric, 'kpi_name')
trace_thresh  = compute_thresholds_full(df_trace, 'trace_name')
log_thresh    = compute_thresholds_full(df_log, 'log_name')

# Merge thresholds into aggregated summaries for reference (and ensure consistent KPI_name column)
# (metric_agg/trace_agg/log_agg already exist and have KPI_name)
metric_groups = metric_agg.merge(metric_thresh, on=['cmdb_id','KPI_name'], how='left', suffixes=('','_th'))
trace_groups  = trace_agg.merge(trace_thresh,  on=['cmdb_id','KPI_name'], how='left', suffixes=('','_th'))
log_groups    = log_agg.merge(log_thresh,    on=['cmdb_id','KPI_name'], how='left', suffixes=('','_th'))

# Function to detect per-point anomalies in window for a raw-window df
low_substrings = ['row_count','ss','success','rate']

def detect_anomalies_per_point(raw_win_df, thresh_df, raw_name_col):
    # raw_win_df has columns: timestamp, cmdb_id, <name_col>, value
    # thresh_df has columns: cmdb_id, KPI_name, p95, p5
    # Normalize column names for merge
    left = raw_win_df.copy()
    left = left.rename(columns={raw_name_col: 'KPI_name'})
    merged = left.merge(thresh_df[['cmdb_id','KPI_name','p95','p5']], on=['cmdb_id','KPI_name'], how='left')
    # Determine low-applicable KPIs
    merged['kpi_lower'] = merged['KPI_name'].str.lower().fillna('')
    merged['low_applicable'] = merged['kpi_lower'].apply(lambda s: any(sub in s for sub in low_substrings))
    # Mark anomalies: high if value >= p95; low if low_applicable and value <= p5
    merged['anomaly_high'] = merged['value'].notna() & merged['p95'].notna() & (merged['value'] >= merged['p95'])
    merged['anomaly_low']  = merged['value'].notna() & merged['p5'].notna()  & merged['low_applicable'] & (merged['value'] <= merged['p5'])
    # Build anomaly rows (one row per anomalous point, with type)
    highs = merged[merged['anomaly_high']].copy()
    if not highs.empty:
        highs['anomaly_type'] = 'high'
    lows = merged[merged['anomaly_low']].copy()
    if not lows.empty:
        lows['anomaly_type'] = 'low'
    anomalies = pd.concat([highs, lows], ignore_index=True, sort=False)
    if anomalies.empty:
        # keep consistent schema
        anomalies = anomalies.reindex(columns=list(merged.columns) + ['anomaly_type'])
    # Add local timestamp representation
    anomalies['timestamp_local'] = pd.to_datetime(anomalies['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # Select compact columns for per-point table
    per_point = anomalies[['cmdb_id','KPI_name','timestamp','timestamp_local','value','anomaly_type','p95','p5']]
    return per_point

# Detect anomalies in each domain
metric_anom_points = detect_anomalies_per_point(metric_win, metric_thresh, 'kpi_name')
trace_anom_points  = detect_anomalies_per_point(trace_win,  trace_thresh,  'trace_name')
log_anom_points    = detect_anomalies_per_point(log_win,    log_thresh,    'log_name')

# Combine all anomaly points into one full table (stored in kernel variable)
anomaly_points = pd.concat([metric_anom_points, trace_anom_points, log_anom_points], ignore_index=True, sort=False)

# Now produce per-group summaries for groups that have >=1 anomaly point in the window
if not anomaly_points.empty:
    # Aggregate per group and anomaly_type
    def agg_group(df):
        # for high: earliest timestamp_local, count, max value; for low: earliest, count, min value
        agg_high = df[df['anomaly_type']=='high'].groupby(['cmdb_id','KPI_name','anomaly_type']).agg(
            earliest_anomaly_ts = ('timestamp_local','min'),
            anomaly_point_count = ('value','count'),
            extremal_value = ('value','max'),
            group_p95 = ('p95','first'),
            group_p5  = ('p5','first')
        ).reset_index()
        agg_low = df[df['anomaly_type']=='low'].groupby(['cmdb_id','KPI_name','anomaly_type']).agg(
            earliest_anomaly_ts = ('timestamp_local','min'),
            anomaly_point_count = ('value','count'),
            extremal_value = ('value','min'),
            group_p95 = ('p95','first'),
            group_p5  = ('p5','first')
        ).reset_index()
        return pd.concat([agg_high, agg_low], ignore_index=True, sort=False)
    anomaly_summary = agg_group(anomaly_points)
    # Compute severity
    def compute_sev(row):
        if row['anomaly_type']=='high':
            p95 = row['group_p95']
            maxv = row['extremal_value']
            try:
                if pd.isna(p95) or p95 == 0:
                    return np.nan
                return float(maxv) / float(p95)
            except Exception:
                return np.nan
        else:
            p5 = row['group_p5']
            minv = row['extremal_value']
            try:
                if pd.isna(minv) or minv == 0:
                    return np.nan
                return float(p5) / float(minv)
            except Exception:
                return np.nan
    anomaly_summary['severity'] = anomaly_summary.apply(compute_sev, axis=1)
    # Rename columns to match required output names and tidy
    anomaly_summary = anomaly_summary.rename(columns={
        'earliest_anomaly_ts': 'earliest_anomaly_timestamp_local',
        'extremal_value': 'max_or_min_value_in_window'
    })
    # Reorder columns
    anomaly_summary = anomaly_summary[[
        'cmdb_id','KPI_name','anomaly_type','earliest_anomaly_timestamp_local',
        'anomaly_point_count','max_or_min_value_in_window','group_p95','group_p5','severity'
    ]]
    # Sort by severity desc and keep top 20
    anomaly_summary = anomaly_summary.sort_values('severity', ascending=False).reset_index(drop=True)
    anomaly_summary_top20 = anomaly_summary.head(20)
else:
    anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','earliest_anomaly_timestamp_local',
        'anomaly_point_count','max_or_min_value_in_window','group_p95','group_p5','severity'
    ])
    anomaly_summary_top20 = anomaly_summary

# Totals
total_groups_scanned = int(metric_agg.shape[0] + trace_agg.shape[0] + log_agg.shape[0])
groups_with_anomalies = int(anomaly_summary[['cmdb_id','KPI_name']].drop_duplicates().shape[0])

# Final compact outputs
anomaly_summary_top20, total_groups_scanned, groups_with_anomalies, anomaly_points.shape

```
Out[2]:
```
```python
# Summarize anomaly detection results in plain English (compact)
# Reuse existing kernel variables: anomaly_summary_top20, total_groups_scanned, groups_with_anomalies, anomaly_points

# Prepare top entries (up to 5) for concise examples
top_n = 5
if 'anomaly_summary_top20' in globals() and not anomaly_summary_top20.empty:
    examples_df = anomaly_summary_top20.head(top_n)
    examples_lines = []
    for i, r in examples_df.iterrows():
        sev = r['severity']
        sev_str = f"{sev:.3f}" if pd.notna(sev) else "NaN"
        p95 = r['group_p95']
        p95_str = f"{p95:.6g}" if pd.notna(p95) else "NaN"
        p5 = r['group_p5']
        p5_str = f"{p5:.6g}" if pd.notna(p5) else "NaN"
        line = (f"{len(examples_lines)+1}. {r['cmdb_id']} | {r['KPI_name']} | {r['anomaly_type']} | "
                f"severity={sev_str} | earliest={r['earliest_anomaly_timestamp_local']} | "
                f"count={int(r['anomaly_point_count'])} | value={r['max_or_min_value_in_window']} | p95={p95_str} | p5={p5_str}")
        examples_lines.append(line)
    examples_text = "\n".join(examples_lines)
else:
    examples_text = "No anomalies detected in the window."

total_scanned = total_groups_scanned if 'total_groups_scanned' in globals() else 0
groups_with_anoms = groups_with_anomalies if 'groups_with_anomalies' in globals() else 0
total_anom_points = int(anomaly_points.shape[0]) if 'anomaly_points' in globals() else 0

summary = (
    f"Anomaly detection summary for window 2021-03-04 06:00 to 2021-03-04 06:30 (UTC+8):\n\n"
    f"- Total distinct component-KPI groups scanned: {total_scanned}\n"
    f"- Groups with >=1 anomaly in window: {groups_with_anoms}\n"
    f"- Total anomalous points found in window: {total_anom_points}\n\n"
    f"Detection rules used:\n"
    f"- High-anomaly: value >= group's global P95 (computed from full series).\n"
    f"- Low-anomaly (for KPIs containing row_count, ss, success, rate): value <= group's global P5.\n\n"
    f"Top anomaly examples (by severity):\n{examples_text}\n\n"
    f"Full per-point anomaly table is stored in kernel variable: anomaly_points\n"
    f"Per-group aggregated anomaly summary (top results) is in: anomaly_summary_top20"
)

# Display the concise plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           KPI_name anomaly_type earliest_anomaly_timestamp_local  anomaly_point_count  max_or_min_value_in_window   group_p95   group_p5   severity
0   Tomcat02                                      log.row_count          low        2021-03-04 06:05:00+08:00                    2                    1.000000   91.000000  70.500000  70.500000
1   apache01                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:01:00+08:00                    2                    1.012784    0.026981   0.004152  37.536238
2       IG02                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:00:00+08:00                    2                    1.263245    0.054000   0.008300  23.393426
3   Tomcat01                                      log.row_count          low        2021-03-04 06:06:00+08:00                    2                    4.000000   89.500000  68.000000  17.000000
4       MG01                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:00:00+08:00                    2                    0.764700    0.052545   0.008300  14.553240
5       IG01                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:00:00+08:00                    2                    0.203674    0.033665   0.008300   6.050022
6   Tomcat03                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:01:00+08:00                    2                    0.133000    0.024695   0.008300   5.385706
7   apache02                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:01:00+08:00                    2                    0.099717    0.020759   0.004200   4.803555
8   apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps         high        2021-03-04 06:04:00+08:00                    1                    1.000000    0.250000   0.000000   4.000000
9       MG02                             OSLinux-CPU_CPU_CPUWio         high        2021-03-04 06:00:00+08:00                    2                    0.627500    0.166300   0.008300   3.773301
10   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps         high        2021-03-04 06:00:00+08:00                    1                    0.016667    0.005000   0.000000   3.333333
11   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead         high        2021-03-04 06:00:00+08:00                    1                    1.066667    0.320000   0.000000   3.333333
12   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps         high        2021-03-04 06:23:00+08:00                    1                    7.000000    2.100000   0.000000   3.333333
13   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...         high        2021-03-04 06:19:00+08:00                    1                    0.400000    0.120000   0.000000   3.333333
14   Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps         high        2021-03-04 06:25:00+08:00                    1                    1.000000    0.300000   0.000000   3.333333
15  Tomcat02                               trace.self.row_count          low        2021-03-04 06:03:00+08:00                    3                    6.000000  128.150000  18.000000   3.000000
16   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps         high        2021-03-04 06:26:00+08:00                    1                    1.000000    0.350000   0.000000   2.857143
17  Tomcat02                        trace.to_MG02.duration_mean         high        2021-03-04 06:01:00+08:00                    2                    0.718500    0.254400   0.018200   2.824295
18      MG02                  trace.from_Tomcat02.duration_mean         high        2021-03-04 06:01:00+08:00                    2                    0.718500    0.254400   0.018200   2.824295
19      MG02                        trace.to_dockerA2.row_count         high        2021-03-04 06:13:00+08:00                    7                   14.000000    5.000000   1.000000   2.800000, 1809, 1809, (20303, 8))```
```

In [ ]:
# In[3]:

# Build fault intervals from per-point anomalies and apply significance filters
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables: anomaly_points, metric_agg, trace_agg, log_agg, total_groups_scanned
# Ensure anomaly_points exists
if 'anomaly_points' not in globals():
    anomaly_points = pd.DataFrame(columns=['cmdb_id','KPI_name','timestamp','timestamp_local','value','anomaly_type','p95','p5'])

# Work on a copy
ap = anomaly_points.copy()

# Ensure timestamp is integer and sort
if not ap.empty:
    ap['timestamp'] = pd.to_numeric(ap['timestamp'], errors='coerce').astype('Int64')
    ap = ap.sort_values(['cmdb_id','KPI_name','anomaly_type','timestamp']).reset_index(drop=True)

# Function to create intervals within a group (consecutive minutes: diff == 60)
def intervals_from_group(df_group):
    if df_group.shape[0] == 0:
        return pd.DataFrame()
    # compute diff in seconds
    diffs = df_group['timestamp'].diff().fillna(9999).astype('Int64')
    # start new interval where diff != 60
    interval_id = (diffs != 60).cumsum()
    df_group = df_group.assign(interval_id=interval_id.values)
    # aggregate per interval
    records = []
    for iid, sub in df_group.groupby('interval_id', sort=True):
        start_ts = int(sub['timestamp'].min())
        end_ts = int(sub['timestamp'].max())
        start_local = pd.to_datetime(start_ts, unit='s', utc=True).tz_convert(tz)
        end_local = pd.to_datetime(end_ts, unit='s', utc=True).tz_convert(tz)
        # duration in minutes inclusive
        duration_minutes = int((end_ts - start_ts) / 60) + 1
        count_pts = int(sub.shape[0])
        anomaly_type = sub['anomaly_type'].iloc[0]
        # group-level thresholds (should be same for all rows in group)
        group_p95 = sub['p95'].dropna().iloc[0] if sub['p95'].dropna().shape[0] > 0 else np.nan
        group_p5  = sub['p5'].dropna().iloc[0]  if sub['p5'].dropna().shape[0]  > 0 else np.nan
        if anomaly_type == 'high':
            extremal_value = sub['value'].max()
            # severity_ratio = max_value / p95
            if pd.isna(group_p95) or group_p95 == 0:
                severity = np.nan
            else:
                severity = float(extremal_value) / float(group_p95)
        else:  # low
            extremal_value = sub['value'].min()
            # severity_ratio = p5 / min_value
            if pd.isna(group_p5) or extremal_value == 0:
                severity = np.nan
            else:
                severity = float(group_p5) / float(extremal_value)
        records.append({
            'cmdb_id': sub['cmdb_id'].iloc[0],
            'KPI_name': sub['KPI_name'].iloc[0],
            'anomaly_type': anomaly_type,
            'fault_start_local': start_local,
            'fault_end_local': end_local,
            'duration_minutes': duration_minutes,
            'anomaly_point_count': count_pts,
            'extremal_value': extremal_value,
            'group_p95': group_p95,
            'group_p5': group_p5,
            'severity_ratio': severity
        })
    return pd.DataFrame.from_records(records)

# Apply to each (cmdb_id, KPI_name, anomaly_type) group
if ap.empty:
    identified_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local','duration_minutes',
        'anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
    ])
else:
    parts = []
    grp_cols = ['cmdb_id','KPI_name','anomaly_type']
    for keys, group in ap.groupby(grp_cols, sort=True):
        intervals_df = intervals_from_group(group)
        if not intervals_df.empty:
            parts.append(intervals_df)
    if parts:
        identified_faults = pd.concat(parts, ignore_index=True, sort=False)
    else:
        identified_faults = pd.DataFrame(columns=[
            'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local','duration_minutes',
            'anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
        ])

# Now filter by significance:
# - For 'high': severity_ratio > 2.0
# - For 'low' : severity_ratio > 1.5
# - Drop intervals with anomaly_point_count == 1 unless severity_ratio > 3.0
if not identified_faults.empty:
    cond_high = (identified_faults['anomaly_type']=='high') & (identified_faults['severity_ratio'] > 2.0)
    cond_low  = (identified_faults['anomaly_type']=='low')  & (identified_faults['severity_ratio'] > 1.5)
    sig_mask = cond_high | cond_low
    sig_df = identified_faults[sig_mask].copy()
    # Drop singleton points unless severity > 3.0
    singleton_mask = sig_df['anomaly_point_count'] == 1
    keep_singleton = singleton_mask & (sig_df['severity_ratio'] > 3.0)
    # final significant faults
    sig_df = pd.concat([sig_df[~singleton_mask], sig_df[keep_singleton]], ignore_index=True, sort=False)
    # Sort by fault_start_local ascending
    sig_df = sig_df.sort_values('fault_start_local', ascending=True).reset_index(drop=True)
else:
    sig_df = identified_faults.copy()

# Store full faults DataFrame in kernel variable 'identified_faults' (already assigned)
# identified_faults contains all intervals; sig_df contains significant ones

# Compute counts and earliest fault
total_groups_scanned = total_groups_scanned if 'total_groups_scanned' in globals() else (int(metric_agg.shape[0] + trace_agg.shape[0] + log_agg.shape[0]) if all(v in globals() for v in ['metric_agg','trace_agg','log_agg']) else 0)
if sig_df.empty:
    groups_with_significant_faults = 0
    earliest_fault_start_local = pd.NaT
else:
    groups_with_significant_faults = sig_df[['cmdb_id','KPI_name']].drop_duplicates().shape[0]
    earliest_fault_start_local = sig_df['fault_start_local'].min()

# Compact output: top 20 significant faults (requested columns)
faults_top20 = sig_df[[
    'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
    'duration_minutes','anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
]].head(20)

# Final displayed variables (compact)
faults_top20, total_groups_scanned, groups_with_significant_faults, earliest_fault_start_local, identified_faults.shape

```
Out[3]:
```
```python
# Plain-English summary of the fault-identification results (compact)
# Reuse existing kernel variables: faults_top20, total_groups_scanned, groups_with_significant_faults, earliest_fault_start_local, identified_faults

import pandas as pd

# Prepare top examples (up to 5) from faults_top20 for concise display
examples = []
if 'faults_top20' in globals() and not faults_top20.empty:
    for _, r in faults_top20.head(5).iterrows():
        sev = r['severity_ratio']
        sev_s = f"{sev:.3f}" if pd.notna(sev) else "NaN"
        examples.append(f"{r['cmdb_id']} | {r['KPI_name']} | {r['anomaly_type']} | start={r['fault_start_local']} | pts={int(r['anomaly_point_count'])} | extremal={r['extremal_value']} | severity={sev_s}")
else:
    examples.append("No significant faults detected.")

total_scanned = total_groups_scanned if 'total_groups_scanned' in globals() else 0
groups_sig = groups_with_significant_faults if 'groups_with_significant_faults' in globals() else (globals().get('groups_with_significant_faults', 0))
earliest = earliest_fault_start_local if 'earliest_fault_start_local' in globals() else pd.NaT
total_intervals = identified_faults.shape[0] if 'identified_faults' in globals() else 0

summary = (
    f"Fault identification summary (incident window 2021-03-04 06:00–06:30 UTC+8):\n\n"
    f"- Total distinct component-KPI groups scanned: {total_scanned}\n"
    f"- Number of groups that produced significant faults: {groups_sig}\n"
    f"- Total fault intervals detected (all intervals, before final top truncation): {total_intervals}\n"
    f"- Earliest significant fault start (UTC+8): {earliest}\n\n"
    f"Significance filters applied:\n"
    f"- 'high' faults kept if severity_ratio > 2.0 (and single-point faults only if severity_ratio > 3.0).\n"
    f"- 'low'  faults kept if severity_ratio > 1.5 (and single-point faults only if severity_ratio > 3.0).\n\n"
    f"Top example significant faults (by start time / severity, up to 5):\n- " + "\n- ".join(examples) + "\n\n"
    f"The full intervals table is stored in kernel variable: identified_faults\n"
    f"A compact view of top results is stored in: faults_top20"
)

# Display the concise plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                         KPI_name anomaly_type         fault_start_local           fault_end_local  duration_minutes  anomaly_point_count  extremal_value   group_p95   group_p5  severity_ratio
0       IG01                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                 1                    1        0.203674    0.033665   0.008300        6.050022
1       MG01                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:00:00+08:00 2021-03-04 06:01:00+08:00                 2                    2        0.764700    0.052545   0.008300       14.553240
2    Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                 1                    1        1.066667    0.320000   0.000000        3.333333
3    Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                 1                    1        0.016667    0.005000   0.000000        3.333333
4       MG02                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                 1                    1        0.627500    0.166300   0.008300        3.773301
5       IG02                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                 1                    1        1.263245    0.054000   0.008300       23.393426
6   apache01                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                 1                    1        1.012784    0.026981   0.004152       37.536238
7       MG01                      trace.to_dockerB2.row_count          low 2021-03-04 06:01:00+08:00 2021-03-04 06:03:00+08:00                 3                    3        1.000000   13.000000   2.000000        2.000000
8   Tomcat03                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                 1                    1        0.133000    0.024695   0.008300        5.385706
9       MG02                      trace.to_dockerB2.row_count          low 2021-03-04 06:01:00+08:00 2021-03-04 06:03:00+08:00                 3                    3        1.000000    9.900000   2.000000        2.000000
10  apache02                           OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                 1                    1        0.099717    0.020759   0.004200        4.803555
11      IG02                      trace.to_Tomcat01.row_count          low 2021-03-04 06:03:00+08:00 2021-03-04 06:05:00+08:00                 3                    3        1.000000    9.200000   2.000000        2.000000
12  Tomcat01                        trace.from_IG02.row_count          low 2021-03-04 06:03:00+08:00 2021-03-04 06:05:00+08:00                 3                    3        1.000000    9.200000   2.000000        2.000000
13  Tomcat02                             trace.self.row_count          low 2021-03-04 06:03:00+08:00 2021-03-04 06:04:00+08:00                 2                    2        6.000000  128.150000  18.000000        3.000000
14  apache02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps         high 2021-03-04 06:04:00+08:00 2021-03-04 06:04:00+08:00                 1                    1        1.000000    0.250000   0.000000        4.000000
15  Tomcat03                                    log.row_count          low 2021-03-04 06:04:00+08:00 2021-03-04 06:06:00+08:00                 3                    3        1.000000   22.200000   2.000000        2.000000
16      MG02                             trace.self.row_count          low 2021-03-04 06:04:00+08:00 2021-03-04 06:05:00+08:00                 2                    2        3.000000   60.850000   5.900000        1.966667
17      MG01                             trace.self.row_count          low 2021-03-04 06:04:00+08:00 2021-03-04 06:05:00+08:00                 2                    2        4.000000   61.650000   6.450000        1.612500
18  Tomcat03                             trace.self.row_count          low 2021-03-04 06:04:00+08:00 2021-03-04 06:06:00+08:00                 3                    3        6.000000  144.500000  15.000000        2.500000
19      IG02                         trace.self.duration_mean         high 2021-03-04 06:07:00+08:00 2021-03-04 06:08:00+08:00                 2                    2        1.481444    0.633244   0.083686        2.339452, 1809, 26, Timestamp('2021-03-04 06:00:00+0800', tz='Asia/Shanghai'), (10441, 11))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure identified_faults exists
if 'identified_faults' not in globals():
    identified_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'duration_minutes','anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
    ])

# Make a working copy
faults = identified_faults.copy()

# Normalize datetime columns to timezone-aware pandas Timestamps (UTC+8)
for col in ['fault_start_local', 'fault_end_local']:
    if col in faults.columns and not faults[col].empty:
        # If already tz-aware, keep; else try to convert
        if not pd.api.types.is_datetime64_any_dtype(faults[col]):
            faults[col] = pd.to_datetime(faults[col], utc=True).dt.tz_convert(tz)
        else:
            # ensure tz-aware and converted to Asia/Shanghai
            faults[col] = pd.to_datetime(faults[col], utc=True).dt.tz_convert(tz)

# Aggregation by component (cmdb_id)
if faults.empty:
    faults_by_component = pd.DataFrame(columns=[
        'cmdb_id','number_of_fault_intervals','earliest_fault_start_local',
        'latest_fault_end_local','max_severity_ratio','median_severity_ratio'
    ])
else:
    grp = faults.groupby('cmdb_id', sort=True)
    agg = grp.agg(
        number_of_fault_intervals = ('KPI_name','count'),
        earliest_fault_start_local = ('fault_start_local','min'),
        latest_fault_end_local     = ('fault_end_local','max'),
        max_severity_ratio         = ('severity_ratio','max'),
        median_severity_ratio      = ('severity_ratio','median')
    ).reset_index()
    # Round severity columns for compactness
    agg['max_severity_ratio'] = pd.to_numeric(agg['max_severity_ratio'], errors='coerce').round(6)
    agg['median_severity_ratio'] = pd.to_numeric(agg['median_severity_ratio'], errors='coerce').round(6)
    # Sort by max_severity_ratio desc and keep top 20
    faults_by_component = agg.sort_values('max_severity_ratio', ascending=False).head(20).reset_index(drop=True)

# Service-level faults subset
if faults.empty:
    service_level_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'anomaly_point_count','extremal_value','severity_ratio'
    ])
else:
    # Case-insensitive matching for service-level indicators
    mask = faults['KPI_name'].str.contains('trace|tomcat|trace.to_|trace.from_', case=False, na=False)
    svc = faults[mask].copy()
    # Select requested columns and sort by severity descending, keep top 10
    service_level_faults = svc[[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'anomaly_point_count','extremal_value','severity_ratio'
    ]].sort_values('severity_ratio', ascending=False).head(10).reset_index(drop=True)

# Final compact outputs (stored in kernel variables as requested)
faults_by_component, service_level_faults

```
Out[4]:
```
```python
# Summarize the aggregation results in plain English (compact)
# Reuse kernel variables: faults_by_component, service_level_faults

import pandas as pd
import numpy as np

# Safeguard variables
fbc = faults_by_component if 'faults_by_component' in globals() else pd.DataFrame()
svc = service_level_faults if 'service_level_faults' in globals() else pd.DataFrame()

num_components_with_faults = int(fbc.shape[0])
earliest_overall = fbc['earliest_fault_start_local'].min() if not fbc.empty else pd.NaT
latest_overall = fbc['latest_fault_end_local'].max() if not fbc.empty else pd.NaT

# Top 5 components by max_severity_ratio for concise examples
top5 = []
if not fbc.empty:
    for _, r in fbc.head(5).iterrows():
        top5.append(f"{r['cmdb_id']}: intervals={int(r['number_of_fault_intervals'])}, "
                    f"max_severity={r['max_severity_ratio']:.3f}, median_severity={r['median_severity_ratio']:.3f}, "
                    f"range={r['earliest_fault_start_local']} to {r['latest_fault_end_local']}")

# Service-level faults up to 10
svc_list = []
if not svc.empty:
    for _, r in svc.iterrows():
        svc_list.append(f"{r['cmdb_id']} | {r['KPI_name']} | {r['anomaly_type']} | "
                        f"start={r['fault_start_local']} end={r['fault_end_local']} pts={int(r['anomaly_point_count'])} "
                        f"extremal={r['extremal_value']} sev={r['severity_ratio']:.3f}")
        if len(svc_list) >= 10:
            break

summary = (
    "Faults-by-component summary (compact):\n\n"
    f"- Components with significant faults: {num_components_with_faults}\n"
    f"- Earliest fault start among components: {earliest_overall}\n"
    f"- Latest fault end among components: {latest_overall}\n\n"
    "Top components by max severity (top 5):\n"
    + ("\n".join([f"{i+1}. {line}" for i,line in enumerate(top5)]) if top5 else "No component faults detected.") 
    + "\n\n"
    "Service-level faults (examples, up to 10):\n"
    + ("\n".join(svc_list) if svc_list else "No service-level faults matching criteria found.")
    + "\n\n"
    "Notes:\n"
    "- 'faults_by_component' and 'service_level_faults' are stored in the kernel for further inspection.\n"
    "- Times are shown in UTC+8."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  number_of_fault_intervals earliest_fault_start_local    latest_fault_end_local  max_severity_ratio  median_severity_ratio
0   Tomcat02                        595  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           70.500000                    1.0
1   apache01                        511  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           37.536238                    1.0
2       IG02                        563  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           23.393426                    1.0
3   Tomcat01                        645  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           17.000000                    1.0
4       MG01                        643  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           14.553240                    1.0
5       IG01                        537  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            6.050022                    1.0
6   Tomcat03                        635  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            5.385706                    1.0
7   apache02                        527  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            4.803555                    1.0
8       MG02                        654  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.773301                    1.0
9    Mysql01                       1671  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.333333                    1.0
10   Mysql02                       1606  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.333333                    1.0
11   Redis02                        586  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.333333                    1.0
12  Tomcat04                        622  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            2.575000                    1.0
13   Redis01                        646  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            2.173729                    1.0,     cmdb_id                           KPI_name anomaly_type         fault_start_local           fault_end_local  anomaly_point_count  extremal_value  severity_ratio
0  Tomcat02               trace.self.row_count          low 2021-03-04 06:03:00+08:00 2021-03-04 06:04:00+08:00                    2        6.000000        3.000000
1  Tomcat02        trace.to_MG02.duration_mean         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                    1        0.718500        2.824295
2      MG02  trace.from_Tomcat02.duration_mean         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                    1        0.718500        2.824295
3      MG02        trace.to_dockerA2.row_count         high 2021-03-04 06:18:00+08:00 2021-03-04 06:21:00+08:00                    4       14.000000        2.800000
4  Tomcat04               trace.self.row_count          low 2021-03-04 06:06:00+08:00 2021-03-04 06:06:00+08:00                    1        6.000000        2.575000
5  Tomcat03               trace.self.row_count          low 2021-03-04 06:04:00+08:00 2021-03-04 06:06:00+08:00                    3        6.000000        2.500000
6      IG02           trace.self.duration_mean         high 2021-03-04 06:07:00+08:00 2021-03-04 06:08:00+08:00                    2        1.481444        2.339452
7      MG02     trace.to_dockerB2.duration_p95         high 2021-03-04 06:18:00+08:00 2021-03-04 06:18:00+08:00                    1        0.771000        2.192864
8  Tomcat04           trace.self.duration_mean         high 2021-03-04 06:10:00+08:00 2021-03-04 06:10:00+08:00                    1        0.066500        2.048338
9  Tomcat04            trace.self.duration_p95         high 2021-03-04 06:10:00+08:00 2021-03-04 06:10:00+08:00                    1        0.529500        2.020530)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing kernel variables where available: df_trace, trace_agg
tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ["Tomcat02","MG02","Tomcat01","IG02","IG01","MG01"]

# Incident window in UTC epoch seconds (inclusive)
start_local = tz.localize(datetime(2021,3,4,6,0,0))
end_local   = tz.localize(datetime(2021,3,4,6,30,0))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch   = int(end_local.astimezone(pytz.UTC).timestamp())

# Ensure df_trace exists
if 'df_trace' not in globals():
    raise FileNotFoundError("df_trace not found in kernel. Please load trace.csv first.")

# Ensure timestamp and value types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Filter by candidates and incident window (inclusive)
filtered_trace_window = df_trace[
    df_trace['cmdb_id'].isin(candidates) &
    (df_trace['timestamp'] >= start_epoch) &
    (df_trace['timestamp'] <= end_epoch)
].copy()

# Prepare raw rows output: selected columns, sorted by timestamp ascending, top 100
filtered_trace_window_display = filtered_trace_window[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(100)

# Build grouped count table per (cmdb_id, trace_name) within window
if filtered_trace_window.empty:
    trace_group_window = pd.DataFrame(columns=['cmdb_id','trace_name','row_count_in_window','earliest_timestamp_local','latest_timestamp_local','p95_global'])
else:
    grp = filtered_trace_window.groupby(['cmdb_id','trace_name'], sort=True)
    grouped = grp.agg(
        row_count_in_window = ('value','count'),
        earliest_ts = ('timestamp','min'),
        latest_ts = ('timestamp','max')
    ).reset_index()
    # Convert earliest/latest to local timezone
    grouped['earliest_timestamp_local'] = pd.to_datetime(grouped['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)
    grouped['latest_timestamp_local']   = pd.to_datetime(grouped['latest_ts'], unit='s', utc=True).dt.tz_convert(tz)
    grouped = grouped.drop(columns=['earliest_ts','latest_ts'])
    # Merge global p95 from trace_agg (precomputed from full series)
    if 'trace_agg' in globals():
        p95_lookup = trace_agg[['cmdb_id','KPI_name','p95']].rename(columns={'KPI_name':'trace_name','p95':'p95_global'})
        trace_group_window = pd.merge(grouped, p95_lookup, on=['cmdb_id','trace_name'], how='left')
    else:
        # If trace_agg missing, set p95_global to NaN
        grouped['p95_global'] = pd.NA
        trace_group_window = grouped
    # Reorder columns and sort by row_count desc, limit top 50
    trace_group_window = trace_group_window[[
        'cmdb_id','trace_name','row_count_in_window','earliest_timestamp_local','latest_timestamp_local','p95_global'
    ]].sort_values('row_count_in_window', ascending=False).head(50).reset_index(drop=True)

# Final compact outputs (variables to be displayed)
filtered_trace_window_display, trace_group_window

```
Out[5]:
```
```python
# Plain-English compact summary of the filtered trace data and grouped counts
# Reuse kernel variables: filtered_trace_window, trace_group_window

if 'filtered_trace_window' not in globals():
    filtered_trace_window = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])
if 'trace_group_window' not in globals():
    trace_group_window = pd.DataFrame(columns=['cmdb_id','trace_name','row_count_in_window','earliest_timestamp_local','latest_timestamp_local','p95_global'])

# Small extracts for the summary
num_filtered_rows = int(filtered_trace_window.shape[0])
num_groups = int(trace_group_window.shape[0])

# Top 3 groups by row_count for concise examples
top_groups = []
if not trace_group_window.empty:
    for _, r in trace_group_window.head(3).iterrows():
        top_groups.append(f"{r['cmdb_id']} | {r['trace_name']} | rows={int(r['row_count_in_window'])} | p95_global={r['p95_global']}")
else:
    top_groups.append("No groups in window.")

# Earliest/latest timestamps seen in the grouped table (UTC+8)
if not trace_group_window.empty:
    earliest_seen = trace_group_window['earliest_timestamp_local'].min()
    latest_seen = trace_group_window['latest_timestamp_local'].max()
else:
    earliest_seen = pd.NaT
    latest_seen = pd.NaT

summary = (
    f"Trace window extraction (2021-03-04 06:00–06:30 UTC+8) summary:\n\n"
    f"- Filtered raw trace rows for candidates: {num_filtered_rows} rows (stored in variable 'filtered_trace_window').\n"
    f"- Distinct (cmdb_id, trace_name) groups in window: {num_groups} (stored in variable 'trace_group_window').\n"
    f"- Earliest timestamp in groups: {earliest_seen}; Latest timestamp in groups: {latest_seen}.\n\n"
    f"Top groups by row count (examples):\n- " + "\n- ".join(top_groups) + "\n\n"
    f"Notes:\n- Raw sample (top 100 rows) is in variable 'filtered_trace_window' (displayed earlier).\n- Grouped counts include global P95 per group from the precomputed trace_agg.\n"
)

# Display the compact plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp   cmdb_id                      trace_name      value
0    1614808800      IG01        trace.self.duration_mean   0.155150
1    1614808800      IG01         trace.self.duration_p95   0.505000
66   1614808800  Tomcat01        trace.self.duration_mean   0.003267
67   1614808800  Tomcat01         trace.self.duration_p95   0.021400
68   1614808800  Tomcat01            trace.self.row_count  30.000000
..          ...       ...                             ...        ...
193  1614808860      MG02  trace.to_dockerB2.duration_p95   0.210300
224  1614808860  Tomcat02         trace.to_MG02.row_count   2.000000
223  1614808860  Tomcat02      trace.to_MG02.duration_p95   1.351650
222  1614808860  Tomcat02     trace.to_MG02.duration_mean   0.718500
221  1614808860  Tomcat02         trace.to_MG01.row_count   1.000000

[100 rows x 4 columns],      cmdb_id                         trace_name  row_count_in_window  earliest_timestamp_local    latest_timestamp_local  p95_global
0       IG01           trace.self.duration_mean                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    1.157668
1       IG01            trace.self.duration_p95                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    3.278387
2       IG01               trace.self.row_count                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00   73.100000
3   Tomcat01               trace.self.row_count                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00  126.650000
4       MG01           trace.self.duration_mean                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.200988
..       ...                                ...                  ...                       ...                       ...         ...
45      MG02   trace.from_Tomcat03.duration_p95                   29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.609840
46      MG02      trace.from_Tomcat04.row_count                   29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00   10.000000
47      MG02  trace.from_Tomcat04.duration_mean                   29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.273240
48      MG02   trace.from_Tomcat04.duration_p95                   29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.532670
49      MG01   trace.from_Tomcat01.duration_p95                   29 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00    0.510380

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: identified_faults, filtered_trace_window
if 'identified_faults' not in globals():
    identified_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'duration_minutes','anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
    ])
if 'filtered_trace_window' not in globals():
    filtered_trace_window = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

# 1) MG02 metric/disk/IO related faults
patterns = ['CPUWio','LOCALDISK','DSK','DISK','IO']
pat_regex = '|'.join(patterns)
mg02_metric_faults = identified_faults[
    (identified_faults['cmdb_id'] == 'MG02') &
    (identified_faults['KPI_name'].str.contains(pat_regex, case=False, na=False))
].copy()

# Ensure fault_start_local and fault_end_local are tz-aware UTC+8
for col in ['fault_start_local','fault_end_local']:
    if col in mg02_metric_faults.columns and not mg02_metric_faults.empty:
        # If not datetime dtype or tz-naive, convert
        if not pd.api.types.is_datetime64_any_dtype(mg02_metric_faults[col]):
            mg02_metric_faults[col] = pd.to_datetime(mg02_metric_faults[col], utc=True).dt.tz_convert(tz)
        else:
            mg02_metric_faults[col] = pd.to_datetime(mg02_metric_faults[col], utc=True).dt.tz_convert(tz)

# Select requested columns and sort/limit
mg02_metric_faults = mg02_metric_faults[[
    'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
    'anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
]].sort_values('fault_start_local', ascending=True).head(20).reset_index(drop=True)

# 2) Tomcat02 <-> MG02 trace rows from filtered_trace_window
# Ensure timestamp is numeric
if not filtered_trace_window.empty:
    filtered_trace_window['timestamp'] = pd.to_numeric(filtered_trace_window['timestamp'], errors='coerce').astype('Int64')

# Build mask for the two directions
mask_t_to_mg = (filtered_trace_window['cmdb_id'] == 'Tomcat02') & (filtered_trace_window['trace_name'].str.contains('to_MG02', case=False, na=False))
mask_mg_from_t = (filtered_trace_window['cmdb_id'] == 'MG02') & (filtered_trace_window['trace_name'].str.contains('from_Tomcat02', case=False, na=False))
mask = mask_t_to_mg | mask_mg_from_t

tomcat02_mg02_traces = filtered_trace_window[mask].copy()

# Add timestamp_local column
if not tomcat02_mg02_traces.empty:
    tomcat02_mg02_traces['timestamp_local'] = pd.to_datetime(tomcat02_mg02_traces['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    tomcat02_mg02_traces = tomcat02_mg02_traces[['timestamp','timestamp_local','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)
else:
    tomcat02_mg02_traces = tomcat02_mg02_traces[['timestamp']].head(0)  # empty with minimal schema

# Final compact outputs (variables displayed)
mg02_metric_faults, tomcat02_mg02_traces

```
Out[6]:
```
```python
# Compact plain-English verification summary for MG02 and Tomcat02->MG02 call chain
# Reuse existing kernel variables: mg02_metric_faults, tomcat02_mg02_traces

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Safeguards
if 'mg02_metric_faults' not in globals():
    mg02_metric_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
    ])
if 'tomcat02_mg02_traces' not in globals():
    tomcat02_mg02_traces = pd.DataFrame(columns=['timestamp','timestamp_local','cmdb_id','trace_name','value'])

# 1) Summary for MG02 metric/IO-related faults
mg = mg02_metric_faults.copy()
mg_count = int(mg.shape[0])

if mg_count > 0:
    # Ensure fault_start_local is datetime for min/max
    if not pd.api.types.is_datetime64_any_dtype(mg['fault_start_local']):
        mg['fault_start_local'] = pd.to_datetime(mg['fault_start_local'], utc=True).dt.tz_convert(tz)
    if not pd.api.types.is_datetime64_any_dtype(mg['fault_end_local']):
        mg['fault_end_local'] = pd.to_datetime(mg['fault_end_local'], utc=True).dt.tz_convert(tz)
    mg_earliest = mg['fault_start_local'].min()
    mg_latest = mg['fault_end_local'].max()
    # Top 3 by severity (descending, NaN last)
    mg_top3 = mg.sort_values('severity_ratio', ascending=False, na_position='last').head(3)
    top3_lines = []
    for _, r in mg_top3.iterrows():
        sev = r['severity_ratio']
        sev_s = f"{sev:.3f}" if pd.notna(sev) else "NaN"
        gp95 = r['group_p95']
        gp5 = r['group_p5']
        top3_lines.append(f"{r['KPI_name']} | type={r['anomaly_type']} | start={r['fault_start_local']} | pts={int(r['anomaly_point_count'])} | extremal={r['extremal_value']} | p95={gp95} p5={gp5} | sev={sev_s}")
    # Count how many with severity > 2.0
    mg_highsev_count = int(mg['severity_ratio'].gt(2.0).sum())
else:
    mg_earliest = pd.NaT
    mg_latest = pd.NaT
    top3_lines = []
    mg_highsev_count = 0

# 2) Summary for Tomcat02 <-> MG02 traces
tr = tomcat02_mg02_traces.copy()
tr_count = int(tr.shape[0])
if tr_count > 0:
    # Ensure timestamp_local exists and is tz-aware
    if 'timestamp_local' in tr.columns and not pd.api.types.is_datetime64_any_dtype(tr['timestamp_local']):
        tr['timestamp_local'] = pd.to_datetime(tr['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    tr_earliest = tr['timestamp_local'].min()
    tr_latest = tr['timestamp_local'].max()
    # Counts by direction
    tomcat_to_mg = int(((tr['cmdb_id']=='Tomcat02') & tr['trace_name'].str.contains('to_MG02', case=False, na=False)).sum())
    mg_from_tomcat = int(((tr['cmdb_id']=='MG02') & tr['trace_name'].str.contains('from_Tomcat02', case=False, na=False)).sum())
    # Example notable trace rows (top 5 by timestamp)
    example_traces = []
    for _, r in tr.sort_values('timestamp').head(5).iterrows():
        example_traces.append(f"{r['timestamp_local']} | {r['cmdb_id']} | {r['trace_name']} = {r['value']}")
else:
    tr_earliest = pd.NaT
    tr_latest = pd.NaT
    tomcat_to_mg = 0
    mg_from_tomcat = 0
    example_traces = []

# Compose concise plain-English summary
summary_lines = [
    "Verification evidence (compact) for MG02 and Tomcat02->MG02 call chain within 2021-03-04 06:00–06:30 (UTC+8):",
    "",
    "1) MG02 metric / disk / IO-related significant faults:",
    f"- Rows returned: {mg_count}",
    f"- Earliest fault start: {mg_earliest}",
    f"- Latest fault end:      {mg_latest}",
    f"- Faults with severity_ratio > 2.0: {mg_highsev_count}",
    "- Top faults by severity (up to 3):"
]
if top3_lines:
    summary_lines += [f"  - {l}" for l in top3_lines]
else:
    summary_lines += ["  - (no MG02 faults found)"]

summary_lines += [
    "",
    "2) Tomcat02 <-> MG02 trace evidence in window (filtered_trace_window):",
    f"- Trace rows returned: {tr_count}",
    f"- Earliest trace timestamp: {tr_earliest}",
    f"- Latest trace timestamp:   {tr_latest}",
    f"- Tomcat02 -> MG02 trace rows: {tomcat_to_mg}",
    f"- MG02 <- Tomcat02 trace rows: {mg_from_tomcat}",
    "- Example trace rows (first few by time):"
]
if example_traces:
    summary_lines += [f"  - {l}" for l in example_traces]
else:
    summary_lines += ["  - (no matching traces found)"]

summary_lines += [
    "",
    "Conclusion (evidence):",
    "- MG02 shows metric-level CPU WIO high anomaly (example severity ~3.77) at ~2021-03-04 06:00,",
    "  and multiple disk/IO metric entries are present (some with NaN/zero thresholds).",
    "- Trace-level evidence shows Tomcat02 -> MG02 calls during the same window with elevated durations",
    "  (e.g., MG02 trace.from_Tomcat02.duration_mean = 0.7185 at 06:01 with severity ~2.82),",
    "  supporting that Tomcat02 -> MG02 call latency is elevated around the incident.",
    "",
    "Stored kernel variables:",
    "- mg02_metric_faults  (filtered MG02 faults returned above)",
    "- tomcat02_mg02_traces (filtered trace rows for Tomcat02<->MG02)"
]

summary = "\n".join(summary_lines)

# Display the compact summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           KPI_name anomaly_type         fault_start_local           fault_end_local  anomaly_point_count  extremal_value  group_p95  group_p5  severity_ratio
0     MG02                             OSLinux-CPU_CPU_CPUWio         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1         0.62750   0.166300  0.008300        3.773301
1     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1         0.20000   0.200000  0.000000        1.000000
2     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1         0.00000   0.000000  0.000000             NaN
3     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1         0.00000   0.000000  0.000000             NaN
4     MG02                            trace.self.duration_p95         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                    1         0.67840   0.609402  0.207670        1.113222
5     MG02                   trace.from_Tomcat02.duration_p95         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                    1         1.35165   0.781520  0.023160        1.729514
6     MG02                  trace.from_Tomcat02.duration_mean         high 2021-03-04 06:01:00+08:00 2021-03-04 06:01:00+08:00                    1         0.71850   0.254400  0.018200        2.824295
7     MG02                  trace.from_Tomcat04.duration_mean         high 2021-03-04 06:02:00+08:00 2021-03-04 06:02:00+08:00                    1         0.27900   0.273240  0.019867        1.021080
8     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...         high 2021-03-04 06:04:00+08:00 2021-03-04 06:04:00+08:00                    1         0.83000   0.830000  0.830000        1.000000
9     MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps         high 2021-03-04 06:04:00+08:00 2021-03-04 06:04:00+08:00                    1         0.00000   0.000000  0.000000             NaN
10    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps         high 2021-03-04 06:04:00+08:00 2021-03-04 06:04:00+08:00                    1         0.00000   0.000000  0.000000             NaN
11    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...         high 2021-03-04 06:04:00+08:00 2021-03-04 06:04:00+08:00                    1         0.95000   0.950000  0.950000        1.000000
12    MG02                  trace.from_Tomcat01.duration_mean         high 2021-03-04 06:05:00+08:00 2021-03-04 06:05:00+08:00                    1         0.26500   0.251167  0.037933        1.055076
13    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead         high 2021-03-04 06:06:00+08:00 2021-03-04 06:06:00+08:00                    1         0.00000   0.000000  0.000000             NaN
14    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps         high 2021-03-04 06:06:00+08:00 2021-03-04 06:06:00+08:00                    1         0.00000   0.000000  0.000000             NaN
15    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...         high 2021-03-04 06:06:00+08:00 2021-03-04 06:06:00+08:00                    1         0.20000   0.200000  0.000000        1.000000
16    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps         high 2021-03-04 06:06:00+08:00 2021-03-04 06:06:00+08:00                    1         0.00000   0.000000  0.000000             NaN
17    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps         high 2021-03-04 06:06:00+08:00 2021-03-04 06:06:00+08:00                    1         0.00000   0.000000  0.000000             NaN
18    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps         high 2021-03-04 06:07:00+08:00 2021-03-04 06:07:00+08:00                    1         2.00000   2.000000  0.000000        1.000000
19    MG02                     trace.to_dockerA2.duration_p95         high 2021-03-04 06:07:00+08:00 2021-03-04 06:07:00+08:00                    1         0.67160   0.634217  0.032960        1.058943,      timestamp           timestamp_local   cmdb_id                         trace_name     value
0   1614808800 2021-03-04 06:00:00+08:00      MG02  trace.from_Tomcat02.duration_mean  0.032000
1   1614808800 2021-03-04 06:00:00+08:00      MG02   trace.from_Tomcat02.duration_p95  0.040800
2   1614808800 2021-03-04 06:00:00+08:00      MG02      trace.from_Tomcat02.row_count  3.000000
3   1614808800 2021-03-04 06:00:00+08:00  Tomcat02        trace.to_MG02.duration_mean  0.032000
4   1614808800 2021-03-04 06:00:00+08:00  Tomcat02         trace.to_MG02.duration_p95  0.040800
..         ...                       ...       ...                                ...       ...
45  1614809280 2021-03-04 06:08:00+08:00  Tomcat02        trace.to_MG02.duration_mean  0.026000
46  1614809280 2021-03-04 06:08:00+08:00  Tomcat02         trace.to_MG02.duration_p95  0.039500
47  1614809280 2021-03-04 06:08:00+08:00  Tomcat02            trace.to_MG02.row_count  2.000000
48  1614809340 2021-03-04 06:09:00+08:00      MG02  trace.from_Tomcat02.duration_mean  0.093333
49  1614809340 2021-03-04 06:09:00+08:00      MG02   trace.from_Tomcat02.duration_p95  0.220800

[50 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing variables where possible
tz = pytz.timezone('Asia/Shanghai')
components = ['Tomcat02','Tomcat01','MG02','MG01','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02','apache01','apache02']

# Ensure df_log is loaded
if 'df_log' not in globals():
    import os
    if os.path.exists('log.csv'):
        df_log = pd.read_csv('log.csv')
    else:
        raise FileNotFoundError("log.csv not found in working directory and df_log not in kernel.")

# Normalize timestamp and value columns
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Incident window (inclusive) in UTC epoch seconds
start_local = tz.localize(datetime(2021,3,4,6,0,0))
end_local   = tz.localize(datetime(2021,3,4,6,30,0))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter log rows by components and window
filtered_log_window = df_log[
    df_log['cmdb_id'].isin(components) &
    (df_log['timestamp'] >= start_epoch) &
    (df_log['timestamp'] <= end_epoch)
].copy()

# Add timestamp_local column (UTC->Asia/Shanghai)
if not filtered_log_window.empty:
    filtered_log_window['timestamp_local'] = pd.to_datetime(filtered_log_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Prepare compact raw display: selected columns, sorted, top 100
filtered_log_window_display = filtered_log_window[['timestamp','timestamp_local','cmdb_id','log_name','value']].sort_values('timestamp', ascending=True).head(100)

# Build grouped log summary per (cmdb_id, log_name) within window
if filtered_log_window.empty:
    log_group_window = pd.DataFrame(columns=['cmdb_id','log_name','row_count_in_window','earliest_timestamp_local','latest_timestamp_local','global_p95_from_log_agg'])
else:
    grp = filtered_log_window.groupby(['cmdb_id','log_name'], sort=True).agg(
        row_count_in_window = ('value','count'),
        earliest_ts = ('timestamp','min'),
        latest_ts = ('timestamp','max')
    ).reset_index()
    grp['earliest_timestamp_local'] = pd.to_datetime(grp['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)
    grp['latest_timestamp_local']   = pd.to_datetime(grp['latest_ts'], unit='s', utc=True).dt.tz_convert(tz)
    grp = grp.drop(columns=['earliest_ts','latest_ts'])
    # Merge global P95 from precomputed log_agg (thresholds must be calculated from full series beforehand)
    if 'log_agg' in globals():
        p95_lookup = log_agg[['cmdb_id','KPI_name','p95']].rename(columns={'KPI_name':'log_name','p95':'global_p95_from_log_agg'})
        log_group_window = pd.merge(grp, p95_lookup, on=['cmdb_id','log_name'], how='left')
    else:
        grp['global_p95_from_log_agg'] = pd.NA
        log_group_window = grp
    # Reorder and sort by row_count_in_window desc, limit top 50
    log_group_window = log_group_window[[
        'cmdb_id','log_name','row_count_in_window','earliest_timestamp_local','latest_timestamp_local','global_p95_from_log_agg'
    ]].sort_values('row_count_in_window', ascending=False).head(50).reset_index(drop=True)

# Final compact outputs (stored in kernel variables as requested)
filtered_log_window_display, log_group_window

```
Out[7]:
```
```python
# Summarize the filtered log results in plain English (compact)
# Reuse kernel variables: filtered_log_window (full), filtered_log_window_display (sample), log_group_window

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Safeguard variables
filtered_full = globals().get('filtered_log_window', pd.DataFrame())
filtered_display = globals().get('filtered_log_window_display', pd.DataFrame())
grouped = globals().get('log_group_window', pd.DataFrame())

# Compute summary stats
num_filtered_rows = int(filtered_full.shape[0])
num_groups = int(grouped.shape[0])

if not grouped.empty:
    earliest = grouped['earliest_timestamp_local'].min()
    latest = grouped['latest_timestamp_local'].max()
    # Top 5 groups by row_count_in_window for examples
    top_examples = []
    for _, r in grouped.head(5).iterrows():
        p95 = r.get('global_p95_from_log_agg', None)
        p95_s = f"{p95:.6g}" if pd.notna(p95) else "NaN"
        top_examples.append(f"{r['cmdb_id']} | {r['log_name']} | rows={int(r['row_count_in_window'])} | p95={p95_s}")
else:
    earliest = pd.NaT
    latest = pd.NaT
    top_examples = []

summary = (
    "Log extraction summary (2021-03-04 06:00–06:30 UTC+8):\n\n"
    f"- Filtered raw log rows (full): {num_filtered_rows} rows (stored in 'filtered_log_window').\n"
    f"- Sample of raw rows (top 100) is in 'filtered_log_window_display'.\n"
    f"- Distinct (cmdb_id, log_name) groups in window: {num_groups} (stored in 'log_group_window').\n"
    f"- Earliest group timestamp: {earliest}\n"
    f"- Latest group timestamp:   {latest}\n\n"
    "Top groups by row count (examples):\n"
    + ("\n".join([f"- {l}" for l in top_examples]) if top_examples else "- (no groups found)\n")
    + "\nNotes:\n- Global P95 per group was merged from the previously computed log_agg into 'log_group_window'.\n"
    "- All times shown are UTC+8. Full tables are stored in the kernel for further inspection."
)

# Display the summary string variable (plain English)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp           timestamp_local   cmdb_id          log_name  value
0    1614808800 2021-03-04 06:00:00+08:00  Tomcat01   log.error_count   0.00
1    1614808800 2021-03-04 06:00:00+08:00  Tomcat01  log.gc_pause_max   5.10
2    1614808800 2021-03-04 06:00:00+08:00  Tomcat01     log.row_count  74.00
3    1614808800 2021-03-04 06:00:00+08:00  Tomcat02   log.error_count   0.00
4    1614808800 2021-03-04 06:00:00+08:00  Tomcat02  log.gc_pause_max   5.11
..          ...                       ...       ...               ...    ...
131  1614809340 2021-03-04 06:09:00+08:00  Tomcat02     log.row_count  72.00
136  1614809340 2021-03-04 06:09:00+08:00  apache01   log.error_count   0.00
137  1614809340 2021-03-04 06:09:00+08:00  apache01     log.row_count  10.00
138  1614809340 2021-03-04 06:09:00+08:00  apache02   log.error_count   0.00
139  1614809340 2021-03-04 06:09:00+08:00  apache02     log.row_count  12.00

[100 rows x 5 columns],     cmdb_id          log_name  row_count_in_window  earliest_timestamp_local    latest_timestamp_local  global_p95_from_log_agg
0  Tomcat01   log.error_count                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                     0.00
1  Tomcat01  log.gc_pause_max                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                     5.14
2  Tomcat01     log.row_count                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                    89.50
3  Tomcat02   log.error_count                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                     0.00
4  Tomcat02     log.row_count                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                    91.00
5  apache02     log.row_count                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                    36.00
6  apache02   log.error_count                   31 2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00                     0.00
7  Tomcat02  log.gc_pause_max                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00                     5.13
8  apache01     log.row_count                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00                    37.50
9  apache01   log.error_count                   30 2021-03-04 06:00:00+08:00 2021-03-04 06:29:00+08:00                     0.00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variable 'identified_faults'
tz = pytz.timezone('Asia/Shanghai')

# Ensure identified_faults exists
if 'identified_faults' not in globals():
    identified_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'duration_minutes','anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
    ])

# Work on a copy
nf = identified_faults.copy()

# Normalize fault_start_local and fault_end_local to timezone-aware UTC+8
for col in ['fault_start_local','fault_end_local']:
    if col in nf.columns and not nf.empty:
        # Convert to datetime; if already tz-aware, ensure conversion to Asia/Shanghai
        nf[col] = pd.to_datetime(nf[col], errors='coerce', utc=True).dt.tz_convert(tz)

# Define inclusion and exclusion patterns
include_patterns = ['OSLinux','CPUWio','LOCALDISK','DSK','DISK','IO']
include_regex = '(' + '|'.join(include_patterns) + ')'
exclude_pattern = 'ZABBIX_Host_Uptime'

# Filter KPI_name by inclusion and exclusion (case-insensitive)
mask_include = nf['KPI_name'].str.contains(include_regex, case=False, na=False)
mask_exclude = nf['KPI_name'].str.contains(exclude_pattern, case=False, na=False)
node_level = nf[mask_include & (~mask_exclude)].copy()

# Aggregate by cmdb_id
if node_level.empty:
    node_level_faults_summary = pd.DataFrame(columns=[
        'cmdb_id','number_of_fault_intervals','earliest_fault_start_local',
        'latest_fault_end_local','max_severity_ratio','median_severity_ratio'
    ])
    distinct_nodes_count = 0
    earliest_overall = pd.NaT
else:
    agg = node_level.groupby('cmdb_id', sort=True).agg(
        number_of_fault_intervals = ('KPI_name','count'),
        earliest_fault_start_local = ('fault_start_local','min'),
        latest_fault_end_local     = ('fault_end_local','max'),
        max_severity_ratio         = ('severity_ratio','max'),
        median_severity_ratio      = ('severity_ratio','median')
    ).reset_index()
    # Round severity columns for compactness
    agg['max_severity_ratio'] = pd.to_numeric(agg['max_severity_ratio'], errors='coerce').round(6)
    agg['median_severity_ratio'] = pd.to_numeric(agg['median_severity_ratio'], errors='coerce').round(6)
    # Sort by earliest_fault_start_local ascending, then max_severity_ratio descending
    agg = agg.sort_values(['earliest_fault_start_local','max_severity_ratio'], ascending=[True, False]).reset_index(drop=True)
    node_level_faults_summary = agg.head(20)
    distinct_nodes_count = int(agg['cmdb_id'].nunique())
    earliest_overall = agg['earliest_fault_start_local'].min()

# Final compact outputs (store the aggregation in kernel variable as requested)
node_level_faults_summary, distinct_nodes_count, earliest_overall

```
Out[8]:
```
```python
# Plain-English summary string based on the node-level faults aggregation results
# Reuse kernel variables: node_level_faults_summary, distinct_nodes_count, earliest_overall

import pandas as pd
import pytz

# Safeguard variables
nl = globals().get('node_level_faults_summary', pd.DataFrame())
distinct = int(globals().get('distinct_nodes_count', globals().get('distinct_nodes_count', 0)))
earliest = globals().get('earliest_overall', pd.NaT)

# Prepare top components by max severity for concise examples
top_by_sev = []
if not nl.empty:
    tmp = nl.sort_values('max_severity_ratio', ascending=False).head(5)
    for _, r in tmp.iterrows():
        top_by_sev.append(f"{r['cmdb_id']} (max_severity={r['max_severity_ratio']}, intervals={int(r['number_of_fault_intervals'])})")

# Format earliest timestamp nicely
earliest_str = str(earliest) if pd.notna(earliest) else "N/A"

summary = (
    "Node-level fault summary (filtered KPIs include OSLinux/CPUWio/LOCALDISK/DSK/DISK/IO; ZABBIX_Host_Uptime excluded):\n\n"
    f"- Distinct nodes with node-level faults: {distinct}\n"
    f"- Earliest fault start among these nodes (UTC+8): {earliest_str}\n\n"
    "Top affected nodes by max severity (examples):\n"
    + ("\n".join([f"- {s}" for s in top_by_sev]) if top_by_sev else "- (no node-level faults found)\n")
    + "\nNotes:\n"
    "- Results are stored in kernel variable 'node_level_faults_summary'.\n"
    "- Times are shown in UTC+8."
)

# Display the compact summary string variable
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  number_of_fault_intervals earliest_fault_start_local    latest_fault_end_local  max_severity_ratio  median_severity_ratio
0   apache01                        505  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           37.536238                    1.0
1       IG02                        528  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           23.393426                    1.0
2       MG01                        569  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00           14.553240                    1.0
3       IG01                        506  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            6.050022                    1.0
4   Tomcat03                        587  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            5.385706                    1.0
5   apache02                        520  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            4.803555                    1.0
6       MG02                        576  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.773301                    1.0
7    Mysql01                        688  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.333333                    1.0
8    Mysql02                        684  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.333333                    1.0
9    Redis02                        547  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            3.333333                    1.0
10  Tomcat02                        543  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            2.824295                    1.0
11  Tomcat04                        572  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            2.248497                    1.0
12  Tomcat01                        594  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            1.845208                    1.0
13   Redis01                        607  2021-03-04 06:00:00+08:00 2021-03-04 06:30:00+08:00            1.666667                    1.0, 14, Timestamp('2021-03-04 06:00:00+0800', tz='Asia/Shanghai'))```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing 'identified_faults' from kernel; if missing, create empty DataFrame with expected schema
if 'identified_faults' not in globals():
    identified_faults = pd.DataFrame(columns=[
        'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
        'duration_minutes','anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
    ])

# Work on a copy to avoid mutating kernel variable
df = identified_faults.copy()

# Ensure datetime columns are timezone-aware in Asia/Shanghai
for col in ['fault_start_local','fault_end_local']:
    if col in df.columns and not df[col].empty:
        # parse and convert to Asia/Shanghai
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True).dt.tz_convert(tz)

# Filter for MySQL nodes and the requested KPI name substrings (case-insensitive)
mysql_nodes = ['Mysql01','Mysql02']
patterns = ['LOCALDISK','DSK','CPUWio','DISK','IO']
pat_regex = '(' + '|'.join(patterns) + ')'

mask = df['cmdb_id'].isin(mysql_nodes) & df['KPI_name'].str.contains(pat_regex, case=False, na=False)
mysql_node_faults = df.loc[mask, [
    'cmdb_id','KPI_name','anomaly_type','fault_start_local','fault_end_local',
    'anomaly_point_count','extremal_value','group_p95','group_p5','severity_ratio'
]].copy()

# Sort by fault_start_local ascending, then severity_ratio descending
# For rows with NaT fault_start_local, they will be placed at the end
mysql_node_faults = mysql_node_faults.sort_values(
    by=['fault_start_local','severity_ratio'],
    ascending=[True, False]
).reset_index(drop=True).head(20)

# Display the compact result (stored in kernel variable 'mysql_node_faults')
mysql_node_faults

```
Out[9]:
```
```python
# Compact plain-English summary for mysql_node_faults
# Reuse kernel variable: mysql_node_faults

import pandas as pd
import numpy as np

df = globals().get('mysql_node_faults', pd.DataFrame()).copy()

num_rows = int(df.shape[0])
distinct_nodes = int(df['cmdb_id'].nunique()) if not df.empty else 0
earliest = df['fault_start_local'].min() if not df.empty else pd.NaT

# Top 5 by severity (descending), show severity or "NaN"
if not df.empty:
    top5 = df.sort_values('severity_ratio', ascending=False, na_position='last').head(5)
    top_lines = []
    for _, r in top5.iterrows():
        sev = r['severity_ratio']
        sev_s = f"{sev:.3f}" if pd.notna(sev) else "NaN"
        top_lines.append(f"{r['cmdb_id']} | {r['KPI_name']} | type={r['anomaly_type']} | start={r['fault_start_local']} | pts={int(r['anomaly_point_count'])} | extremal={r['extremal_value']} | sev={sev_s}")
else:
    top_lines = []

summary = (
    f"MySQL node faults (filtered for Mysql01/Mysql02 and disk/IO-related KPIs):\n\n"
    f"- Rows returned: {num_rows}\n"
    f"- Distinct MySQL nodes affected: {distinct_nodes}\n"
    f"- Earliest fault start (UTC+8): {earliest}\n\n"
    "Key observations:\n"
    "- Most faults are on Mysql01 at 2021-03-04 06:00 (UTC+8). Examples:\n"
    + ("\n".join([f"  - {l}" for l in top_lines]) if top_lines else "  - (no faults found)\n")
    + "\n"
    "- Notable: DSKRTps and DSKRead on Mysql01 show severity ~3.33 (global P95 low, so observed values exceed P95 by ~3.33x).\n"
    "- Several other LOCALDISK/DSK metrics on Mysql01 show elevated write/read values (e.g., high DSKWrite, DSKWTps) at the same timestamp.\n"
    "- Some intervals have NaN severity (often due to zero or missing group thresholds), please treat those with caution.\n\n"
    "Conclusion: Disk I/O-related anomalies are concentrated on Mysql01 at the incident start (06:00+08). Mysql02 shows fewer/less-severe related entries.\n\n"
    "The full compact table is stored in kernel variable 'mysql_node_faults'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           KPI_name anomaly_type         fault_start_local           fault_end_local  anomaly_point_count  extremal_value   group_p95     group_p5  severity_ratio
0   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.016667     0.00500     0.000000        3.333333
1   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        1.066667     0.32000     0.000000        3.333333
2   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        5.550000     4.31331     0.744990        1.286715
3   Mysql01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1      157.600000   134.54669    12.133323        1.171341
4   Mysql01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1       14.400000    12.90669     8.546677        1.115700
5   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.966667     0.92000     0.650000        1.050725
6   Mysql01           Mysql-MySQL_3306_Created Tmp Disk Tables         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.033300     0.03330     0.033300        1.000000
7   Mysql01              Mysql-MySQL_3306_Max Used Connections         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1      108.000000   108.00000   108.000000        1.000000
8   Mysql01                    Mysql-MySQL_3306_MaxConnections         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1     2000.000000  2000.00000  2000.000000        1.000000
9   Mysql02           Mysql-MySQL_3306_Created Tmp Disk Tables         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.033300     0.03330     0.032800        1.000000
10  Mysql02              Mysql-MySQL_3306_Max Used Connections         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1       37.000000    37.00000    37.000000        1.000000
11  Mysql02                    Mysql-MySQL_3306_MaxConnections         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1     2000.000000  2000.00000  2000.000000        1.000000
12  Mysql01             Mysql-MySQL_3306_Binlog Cache Disk Use         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
13  Mysql01        Mysql-MySQL_3306_Binlog stmt cache disk use         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
14  Mysql01          Mysql-MySQL_3306_Opened table definitions         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
15  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
16  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
17  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
18  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN
19  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead         high 2021-03-04 06:00:00+08:00 2021-03-04 06:00:00+08:00                    1        0.000000     0.00000     0.000000             NaN```
```